# ClinVar RAG — query & summarize

Pipeline: **ingest** → **index** → **search** → **summarize** (search and summarize run in this notebook).

Prerequisites: `python src/ingest_clinvar.py` then `python src/build_clinvar_index.py`. EDA: `clinvar_eda.ipynb`.

In [ ]:
# Optional: install dependencies if needed
# !pip install chromadb sentence-transformers transformers huggingface_hub accelerate

In [ ]:
# Natural-language query
query = "breast cancer hereditary pathogenic"

In [ ]:
# Retrieval limits
TOP_K_CHUNKS = 20  # chunk pool from Chroma; used to rank variants (not sent to LLM)
TOP_K_RESULTS = 3  # top variants by best-chunk distance; LLM gets 1 full parquet row each

In [ ]:
# Imports
from pathlib import Path
import sys

from sentence_transformers import SentenceTransformer


In [ ]:
# Project paths and shared RAG helpers
project_root = Path("..").resolve()
sys.path.insert(0, str(project_root / "src"))

from rag_search import search
from rag_summary import load_llm, summarize


In [ ]:
# Paths, embedding model, search config, LLM prompt
data_processed = project_root / "data/processed"
chroma_dir = project_root / "data/chroma_db"

EMBED_MODEL = "BAAI/bge-small-en-v1.5"
CLINVAR_PARQUET = data_processed / "clinvar_rag.parquet"
CLINVAR_COLLECTION = "clinvar_chunks"
CLINVAR_RECORD_FIELDS = [
    "VariationID", "Name", "GeneSymbol", "Chromosome", "Start", "Stop",
    "Type", "ClinicalSignificance", "ReviewStatus", "PhenotypeList",
    "PhenotypeIDS", "Assembly", "HGNC_ID", "NumberSubmitters", "LastEvaluated",
]

# Maps Chroma chunk metadata and parquet columns to rag_search.search()
SEARCH_CONFIG = {
    "collection_name": CLINVAR_COLLECTION,  # Chroma collection to query
    "parquet_path": CLINVAR_PARQUET,        # full records joined onto retrieved chunks
    "group_key": "variation_id",            # metadata field: collapse chunks to one variant
    "record_id_column": "VariationID",      # parquet column used to look up full records
    "record_id_meta_key": "variation_id",   # chunk metadata field holding the same ID
    "record_fields": CLINVAR_RECORD_FIELDS, # parquet columns included in LLM context
    "section_header_template": "### Result {rank} — variation_id={variation_id}, gene={gene}",  # per-result header; placeholders from metadata
    "hit_fields": [("variation_id", "variation_id"), ("gene", "gene")],  # metadata keys printed for each hit
    "full_record_label": "Full ClinVar record",       # label above the joined parquet row
    "selected_label": "variant record(s) for context",  # verbose log when loading records
    "record_id_cast": int,  # cast metadata ID before parquet lookup (VariationID is int)
}

# Needs ~8GB+ RAM: LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"
# Needs ~4GB+ RAM: LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
LLM_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_NEW_TOKENS = 512

SYSTEM_PROMPT = (
    "You are a clinical genomics assistant. Summarize ClinVar variant search results "
    "for the user's query using ONLY the full records provided. "
    "Be concise, accurate, and structured. For each relevant variant, mention gene, "
    "variant name, clinical significance, phenotypes, review status, and variation ID. "
    "If information is missing, say so. Do not invent facts."
)


## Search

Embed query → retrieve chunks from Chroma → rank variants → build LLM context.

In [ ]:
embed_model = SentenceTransformer(EMBED_MODEL)
# returns ranked hits + context string for the LLM
result = search(
    query,
    SEARCH_CONFIG,
    top_k_chunks=TOP_K_CHUNKS,
    top_k_results=TOP_K_RESULTS,
    chroma_dir=chroma_dir,
    embed_model=embed_model,
)

## Summarize

Load LLM (once per kernel session) → generate answer from retrieved context only.

- **Download** once → cached under `~/.cache/huggingface/`.
- **RAM:** 0.5B ~2GB · re-run this cell reuses the loaded model.

In [ ]:
# load_llm reuses llm_model/llm_tokenizer if already in RAM
llm_model, llm_tokenizer = load_llm(
    LLM_MODEL,
    model=globals().get("llm_model"),
    tokenizer=globals().get("llm_tokenizer"),
)


In [ ]:
summary = summarize(
    query,
    result["context"],
    llm_model,
    llm_tokenizer,
    SYSTEM_PROMPT,
    max_new_tokens=MAX_NEW_TOKENS,
)
print("=" * 60)
print(f"Query: {query}\n")
print(summary)



The query "breast cancer hereditary pathogenic" was answered with two variants in the ClinVar database:

1. **NM_007294.4(BRCA1)**: A pathogenic variant at position 241 on chromosome 17 that changes the amino acid from valine to glutamine at codon 81. This variant has been reported as causing hereditary breast ovarian cancer syndrome.

2. **NM_007294.4(BRCA1)**: A pathogenic variant at position 5251 on chromosome 17 that changes the amino acid from arginine to cysteine at codon 1751. This variant has been reported as causing hereditary breast ovarian cancer syndrome.

Both variants have been reviewed by an expert panel and are considered pathogenic based on the available evidence.


The user's query "breast cancer hereditary" was answered with two variants in the ClinVar database:

1. **Variation ID: 54604** - This variant is associated with **Benign** genetic changes in the **BRCA1** gene on chromosome 17. The clinical significance is that it may be associated with **Hereditary breast ovarian cancer syndrome**.

2. **Variation ID: 41803** - This variant is also associated with **Benign** genetic changes in the **BRCA1** gene on chromosome 17. It has been reported as being associated with **Hereditary breast ovarian cancer syndrome**.

Both variants have been reviewed by an expert panel and are listed under the **Hereditary breast ovarian cancer syndrome** category.